# 05I — Frozen-Model Interpretation and Publication Figures

**Build ID:** `05I_V1_FROZEN_MODEL_INTERPRETATION_PUBLICATION_FIGURES`

05C established the leading incident-held-out probability model, 05E froze it, 05F serialized the full-data artifact, 05G audited observation-process dependence, and 05H audited forward temporal generalization.

05I interprets—but does not change—the frozen model. It:

- exactly refits the five frozen 05C outer-fold models;
- verifies regenerated OOF probabilities against the saved 05C primary OOF table;
- computes TreeSHAP contributions only on held-out units;
- samples a maximum of 50 units per incident and aggregates importance equally across incidents;
- quantifies feature and source-group stability across outer folds;
- integrates the 05G proxy and 05H temporal robustness results;
- exports manuscript-oriented interpretation figures.

All importance and effect displays describe predictive associations. They are not causal effects of suppression, line construction, weather, fuels, or terrain. No feature selection, tuning, calibration fitting, or threshold selection occurs here.

## 0. Configuration

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "05I_V1_FROZEN_MODEL_INTERPRETATION_PUBLICATION_FIGURES"

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))
MASTER_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_table_assembly_v1" / "fireline_hold_modeling_master_v1.parquet"
UNIT_DESIGN_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_design_v1" / "unit_outer_fold_and_training_weights_v1.parquet"
FEATURE_SETS_PATH = PROJECT_ROOT / "outputs" / "modeling_design_v1" / "FEATURE_SETS_V1.json"
COLUMN_MANIFEST_PATH = PROJECT_ROOT / "outputs" / "modeling_table_assembly_v1" / "MODELING_COLUMN_MANIFEST_V1.csv"
PRIMARY_OOF_PATH = PROJECT_ROOT / "data" / "processed" / "baseline_modeling_v1" / "oof_predictions" / "E_FULL_EXTENDED__row.parquet"
PRIMARY_PARAMS_PATH = PROJECT_ROOT / "models" / "fireline_hold_primary_v1" / "fireline_hold_primary_v1_params.json"
FULL_GAIN_PATH = PROJECT_ROOT / "outputs" / "final_primary_model_v1" / "FINAL_MODEL_FEATURE_IMPORTANCE_V1.csv"
PROXY_FEATURES_PATH = PROJECT_ROOT / "outputs" / "observation_process_proxy_sensitivity_v1" / "OBSERVATION_PROCESS_PROXY_FEATURES_V1.csv"
PROXY_METRICS_PATH = PROJECT_ROOT / "outputs" / "observation_process_proxy_sensitivity_v1" / "OBSERVATION_PROCESS_SENSITIVITY_METRICS_V1.csv"
TEMPORAL_METRICS_PATH = PROJECT_ROOT / "outputs" / "forward_temporal_generalization_v1" / "FORWARD_TEMPORAL_METRICS_V1.csv"
TEMPORAL_BOOTSTRAP_PATH = PROJECT_ROOT / "outputs" / "forward_temporal_generalization_v1" / "FORWARD_TEMPORAL_PAIRED_BOOTSTRAP_V1.csv"
TEMPORAL_PREDICTIONS_PATH = PROJECT_ROOT / "data" / "processed" / "forward_temporal_generalization_v1" / "FORWARD_TEMPORAL_PREDICTIONS_V1.parquet"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "frozen_model_interpretation_v1"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "frozen_model_interpretation_v1"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
for p in [OUTPUT_ROOT, PROCESSED_ROOT, FIGURE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

required_paths = [
    MASTER_PATH, UNIT_DESIGN_PATH, FEATURE_SETS_PATH, COLUMN_MANIFEST_PATH, PRIMARY_OOF_PATH,
    PRIMARY_PARAMS_PATH, FULL_GAIN_PATH, PROXY_FEATURES_PATH, PROXY_METRICS_PATH,
    TEMPORAL_METRICS_PATH, TEMPORAL_BOOTSTRAP_PATH, TEMPORAL_PREDICTIONS_PATH,
]
for p in required_paths:
    assert p.exists(), p

EXPECTED_UNITS = 304_143
EXPECTED_INCIDENTS = 381
OUTER_FOLDS = 5
MODEL_SEED = 20260825
SHAP_SEED = 20260825
SHAP_MAX_UNITS_PER_INCIDENT = 50
TOP_FEATURES_SAVED = 30
TOP_FEATURES_BAR = 25
TOP_FEATURES_SUMMARY = 15
OOF_REPRODUCTION_ATOL = 1e-10
SHAP_ADDITIVITY_ATOL = 1e-6
PRED_EPS = 1e-6

FIXED_PARAMS = {
    "objective": "binary",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 150,
    "subsample": 0.85,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "max_bin": 127,
    "n_jobs": -1,
    "verbosity": -1,
}

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "lightgbm": "lightgbm",
    "matplotlib": "matplotlib",
}
for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

from datetime import datetime, timezone
import gc
import json
import time

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss
from lightgbm import LGBMClassifier
import lightgbm as lgb
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 420)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 240, "axes.spines.top": False, "axes.spines.right": False})

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)
print("LightGBM:", lgb.__version__)

## 1. Frozen feature contract and scientific group definitions

The modeling-column manifest is the authoritative source label. Observation-process variables identified in 05G are separated from the physical approach group. All 803 features must map to exactly one source group and one descriptive family.

In [ ]:
with FEATURE_SETS_PATH.open("r", encoding="utf-8") as f:
    feature_sets = json.load(f)
with PRIMARY_PARAMS_PATH.open("r", encoding="utf-8") as f:
    saved_primary_params = json.load(f)

features = feature_sets["E_FULL_EXTENDED"]
assert len(features) == 803
assert len(features) == len(set(features))
assert saved_primary_params.get("random_state") == MODEL_SEED
assert {k: v for k, v in saved_primary_params.items() if k != "random_state"} == FIXED_PARAMS
assert "subsample_freq" not in FIXED_PARAMS

manifest = pd.read_csv(COLUMN_MANIFEST_PATH)
manifest_features = manifest[manifest["column"].isin(features)][["column", "source"]].copy()
assert len(manifest_features) == 803
assert manifest_features["column"].is_unique
source_by_feature = manifest_features.set_index("column")["source"].to_dict()

proxy_features = pd.read_csv(PROXY_FEATURES_PATH)["excluded_proxy_feature"].astype(str).tolist()
proxy_set = set(proxy_features)
assert len(proxy_features) == 11
assert proxy_set.issubset(set(features))

def source_group(feature):
    source = source_by_feature[feature]
    if feature in proxy_set:
        return "observation_process"
    return {
        "approach": "fire_approach_physical",
        "hrrr": "weather",
        "terrain": "terrain",
        "landfire": "fuels_canopy",
    }[source]

def feature_family(feature):
    group = source_group(feature)
    name = feature.lower()
    if group == "observation_process":
        return "observation_process"
    if group == "fire_approach_physical":
        if "eta" in name:
            return "approach_eta"
        if any(t in name for t in ["angle", "azimuth", "orientation"]):
            return "approach_geometry"
        if any(t in name for t in ["rate", "trend", "r2"]):
            return "approach_rate_trend"
        if any(t in name for t in ["distance", "inside", "sample_points"]):
            return "approach_distance"
        return "approach_other"
    if group == "weather":
        if "rh2" in name:
            return "weather_relative_humidity"
        if "t2" in name or "temperature" in name:
            return "weather_temperature"
        if any(t in name for t in ["wind", "gust", "u10", "v10", "toward", "cross"]):
            return "weather_wind"
        if any(t in name for t in ["precip", "prate", "apcp"]):
            return "weather_precipitation"
        return "weather_other"
    if group == "terrain":
        if "elevation" in name:
            return "terrain_elevation"
        if "slope" in name:
            return "terrain_slope"
        if any(t in name for t in ["aspect", "northness", "eastness"]):
            return "terrain_aspect"
        return "terrain_shape_context"
    if "fbfm" in name:
        return "fuels_fire_behavior_model"
    if any(t in name for t in ["crown", "cbh", "cbd", "cfl", "_cc_", "_ch_"]):
        return "fuels_canopy_structure"
    return "fuels_other"

feature_metadata = pd.DataFrame({"feature": features})
feature_metadata["manifest_source"] = feature_metadata["feature"].map(source_by_feature)
feature_metadata["source_group"] = feature_metadata["feature"].map(source_group)
feature_metadata["feature_family"] = feature_metadata["feature"].map(feature_family)
assert feature_metadata[["manifest_source", "source_group", "feature_family"]].notna().all().all()
assert feature_metadata["source_group"].value_counts().to_dict() == {
    "fuels_canopy": 547, "weather": 148, "fire_approach_physical": 60, "terrain": 37, "observation_process": 11
}
display(feature_metadata["source_group"].value_counts().rename_axis("source_group").reset_index(name="features"))
feature_metadata.to_csv(OUTPUT_ROOT / "INTERPRETATION_FEATURE_GROUPS_V1.csv", index=False)
print("FROZEN FEATURE CONTRACT: PASS")

## 2. Load the frozen design and numeric feature matrix

The same row order, incident folds, feature order, float32 conversion, and fold-specific seed rule used in 05C are retained. Ordinary row weighting is equivalent to an all-ones training-weight vector.

In [ ]:
design = pd.read_parquet(UNIT_DESIGN_PATH)
primary_saved = pd.read_parquet(PRIMARY_OOF_PATH, columns=["unit_uid", "pred_prob"])
feature_table = pd.read_parquet(MASTER_PATH, columns=["unit_uid", *features])

assert len(design) == EXPECTED_UNITS
assert design["unit_uid"].is_unique
assert design["_irwin_norm"].nunique() == EXPECTED_INCIDENTS
assert set(design["outer_fold"].unique()) == set(range(OUTER_FOLDS))
assert (design.groupby("_irwin_norm")["outer_fold"].nunique() == 1).all()
assert len(primary_saved) == EXPECTED_UNITS and primary_saved["unit_uid"].is_unique
assert len(feature_table) == EXPECTED_UNITS and feature_table["unit_uid"].is_unique

base_columns = ["unit_uid", "_irwin_norm", "IncidentName", "Year", "hold_label", "outer_fold"]
frame = design[base_columns].merge(feature_table, on="unit_uid", how="left", validate="one_to_one")
del feature_table
gc.collect()

X = frame[features].apply(pd.to_numeric, errors="coerce").astype(np.float32)
y = frame["hold_label"].to_numpy(np.int8)
folds = frame["outer_fold"].to_numpy(int)
saved_oof = primary_saved.set_index("unit_uid").loc[frame["unit_uid"], "pred_prob"].to_numpy(float)

assert X.shape == (EXPECTED_UNITS, 803)
assert np.isfinite(y).all()
print("Feature matrix GiB:", round(X.memory_usage(index=True, deep=True).sum() / (1024 ** 3), 3))

## 3. Exact outer-fold reproduction and held-out TreeSHAP

Each model is fitted without its outer-test wildfires. TreeSHAP is calculated only for sampled outer-test units. Sampling is capped per incident, and the final importance is the mean of incident-level mean absolute contributions so that large fires do not dominate interpretation.

In [ ]:
def clip_prob(p):
    return np.clip(np.asarray(p, dtype=float), PRED_EPS, 1.0 - PRED_EPS)

def incident_balanced_feature_mean(values, incident_ids, absolute=False):
    x = np.abs(values) if absolute else values
    means = []
    for incident in pd.unique(incident_ids):
        means.append(np.mean(x[np.asarray(incident_ids) == incident], axis=0))
    return np.mean(np.vstack(means), axis=0)

regenerated_oof = np.full(EXPECTED_UNITS, np.nan, dtype=np.float64)
sample_metadata_frames = []
sample_value_blocks = []
sample_shap_blocks = []
fold_importance_rows = []
runtime_rows = []
max_shap_additivity_error = 0.0

for outer_fold in range(OUTER_FOLDS):
    train_idx = np.flatnonzero(folds != outer_fold)
    test_idx = np.flatnonzero(folds == outer_fold)
    params = dict(FIXED_PARAMS)
    params["random_state"] = MODEL_SEED + outer_fold
    model = LGBMClassifier(**params)

    started = time.time()
    row_weights = np.ones(len(train_idx), dtype=float)
    model.fit(X.iloc[train_idx], y[train_idx], sample_weight=row_weights)
    fit_seconds = time.time() - started
    regenerated_oof[test_idx] = clip_prob(model.predict_proba(X.iloc[test_idx])[:, 1])

    rng = np.random.default_rng(SHAP_SEED + outer_fold)
    sampled_indices = []
    test_meta = frame.iloc[test_idx]
    for _, g in test_meta.groupby("_irwin_norm", sort=True):
        candidates = g.index.to_numpy(int)
        if len(candidates) > SHAP_MAX_UNITS_PER_INCIDENT:
            candidates = rng.choice(candidates, size=SHAP_MAX_UNITS_PER_INCIDENT, replace=False)
        sampled_indices.extend(candidates.tolist())
    sampled_indices = np.array(sorted(sampled_indices), dtype=int)

    sample_pred = clip_prob(model.predict_proba(X.iloc[sampled_indices])[:, 1])
    contributions = model.booster_.predict(X.iloc[sampled_indices], pred_contrib=True)
    assert contributions.shape == (len(sampled_indices), len(features) + 1)
    shap_values_fold = np.asarray(contributions[:, :-1], dtype=np.float64)
    base_values_fold = np.asarray(contributions[:, -1], dtype=np.float64)
    shap_probability = expit(np.sum(shap_values_fold, axis=1) + base_values_fold)
    fold_additivity_error = float(np.max(np.abs(shap_probability - sample_pred)))
    max_shap_additivity_error = max(max_shap_additivity_error, fold_additivity_error)

    sample_meta = frame.loc[sampled_indices, base_columns].copy()
    sample_meta["pred_prob"] = sample_pred
    sample_meta["shap_base_value"] = base_values_fold
    sample_metadata_frames.append(sample_meta)
    sample_value_blocks.append(X.iloc[sampled_indices].to_numpy(dtype=np.float32, na_value=np.nan))
    sample_shap_blocks.append(shap_values_fold)

    fold_balanced_importance = incident_balanced_feature_mean(
        shap_values_fold, sample_meta["_irwin_norm"].to_numpy(), absolute=True
    )
    for feature, value in zip(features, fold_balanced_importance):
        fold_importance_rows.append({
            "outer_fold": outer_fold,
            "feature": feature,
            "mean_abs_oof_shap_incident_balanced": float(value),
        })

    runtime_rows.append({
        "outer_fold": outer_fold,
        "train_units": len(train_idx),
        "test_units": len(test_idx),
        "test_incidents": test_meta["_irwin_norm"].nunique(),
        "shap_sample_units": len(sampled_indices),
        "fit_seconds": fit_seconds,
        "shap_additivity_max_abs_error": fold_additivity_error,
    })
    print(f"Fold {outer_fold}: fit={fit_seconds/60:.3f} min | SHAP units={len(sampled_indices):,} | additivity={fold_additivity_error:.3e}")
    del model, row_weights, contributions, shap_values_fold, base_values_fold
    gc.collect()

runtime = pd.DataFrame(runtime_rows)
oof_reproduction_max_abs_error = float(np.max(np.abs(regenerated_oof - saved_oof)))
sample_metadata = pd.concat(sample_metadata_frames, ignore_index=True)
sample_values = np.vstack(sample_value_blocks)
sample_shap = np.vstack(sample_shap_blocks)
fold_importance = pd.DataFrame(fold_importance_rows)
fold_importance["rank_within_fold"] = fold_importance.groupby("outer_fold")["mean_abs_oof_shap_incident_balanced"].rank(method="min", ascending=False).astype(int)

assert np.isfinite(regenerated_oof).all()
assert sample_metadata["unit_uid"].is_unique
assert sample_metadata["_irwin_norm"].nunique() == EXPECTED_INCIDENTS
assert sample_shap.shape == sample_values.shape
assert np.isfinite(sample_shap).all()

runtime.to_csv(OUTPUT_ROOT / "OOF_SHAP_RUNTIME_V1.csv", index=False)
fold_importance.to_csv(OUTPUT_ROOT / "OOF_SHAP_FEATURE_IMPORTANCE_BY_FOLD_V1.csv", index=False)
sample_metadata.to_parquet(PROCESSED_ROOT / "OOF_SHAP_SAMPLE_METADATA_V1.parquet", index=False)
display(runtime)
print("OOF reproduction max abs error:", oof_reproduction_max_abs_error)
print("SHAP sample units:", len(sample_metadata))

## 4. Incident-balanced feature, family, and source importance

Global importance first averages absolute SHAP within each wildfire and then averages equally across the 381 wildfires. Correlated predictors can share or exchange importance, so source groups, families, and fold stability are emphasized over isolated ranks.

In [ ]:
global_mean_abs = incident_balanced_feature_mean(
    sample_shap, sample_metadata["_irwin_norm"].to_numpy(), absolute=True
)
global_mean_signed = incident_balanced_feature_mean(
    sample_shap, sample_metadata["_irwin_norm"].to_numpy(), absolute=False
)

feature_importance = pd.DataFrame({
    "feature": features,
    "mean_abs_oof_shap_incident_balanced": global_mean_abs,
    "mean_signed_oof_shap_incident_balanced": global_mean_signed,
    "sample_missing_fraction": np.mean(~np.isfinite(sample_values), axis=0),
})
feature_importance["rank_oof_shap"] = feature_importance["mean_abs_oof_shap_incident_balanced"].rank(method="min", ascending=False).astype(int)
feature_importance["oof_shap_fraction"] = feature_importance["mean_abs_oof_shap_incident_balanced"] / feature_importance["mean_abs_oof_shap_incident_balanced"].sum()

fold_stats = (
    fold_importance.groupby("feature", as_index=False)
    .agg(
        fold_mean_abs_shap_mean=("mean_abs_oof_shap_incident_balanced", "mean"),
        fold_mean_abs_shap_sd=("mean_abs_oof_shap_incident_balanced", "std"),
        fold_rank_mean=("rank_within_fold", "mean"),
        fold_rank_sd=("rank_within_fold", "std"),
        top_20_fold_count=("rank_within_fold", lambda x: int(np.sum(np.asarray(x) <= 20))),
    )
)
rank_wide = fold_importance.pivot(index="feature", columns="outer_fold", values="rank_within_fold")
rank_wide.columns = [f"rank_fold_{int(c)}" for c in rank_wide.columns]
rank_wide = rank_wide.reset_index()

full_gain = pd.read_csv(FULL_GAIN_PATH)
feature_importance = (
    feature_importance
    .merge(feature_metadata, on="feature", validate="one_to_one")
    .merge(fold_stats, on="feature", validate="one_to_one")
    .merge(rank_wide, on="feature", validate="one_to_one")
    .merge(full_gain, on="feature", how="left", validate="one_to_one")
    .sort_values("rank_oof_shap")
)
assert len(feature_importance) == 803
assert feature_importance["importance_gain_fraction"].notna().all()

source_importance = (
    feature_importance.groupby("source_group", as_index=False)
    .agg(
        features=("feature", "size"),
        total_mean_abs_oof_shap=("mean_abs_oof_shap_incident_balanced", "sum"),
        total_full_data_gain_fraction=("importance_gain_fraction", "sum"),
    )
    .sort_values("total_mean_abs_oof_shap", ascending=False)
)
source_importance["oof_shap_fraction"] = source_importance["total_mean_abs_oof_shap"] / source_importance["total_mean_abs_oof_shap"].sum()

family_importance = (
    feature_importance.groupby(["source_group", "feature_family"], as_index=False)
    .agg(features=("feature", "size"), total_mean_abs_oof_shap=("mean_abs_oof_shap_incident_balanced", "sum"))
    .sort_values("total_mean_abs_oof_shap", ascending=False)
)
family_importance["oof_shap_fraction"] = family_importance["total_mean_abs_oof_shap"] / family_importance["total_mean_abs_oof_shap"].sum()

feature_importance.to_csv(OUTPUT_ROOT / "OOF_SHAP_FEATURE_IMPORTANCE_V1.csv", index=False)
source_importance.to_csv(OUTPUT_ROOT / "OOF_SHAP_SOURCE_GROUP_IMPORTANCE_V1.csv", index=False)
family_importance.to_csv(OUTPUT_ROOT / "OOF_SHAP_FEATURE_FAMILY_IMPORTANCE_V1.csv", index=False)

top_features = feature_importance.head(TOP_FEATURES_SAVED)["feature"].tolist()
feature_position = {feature: i for i, feature in enumerate(features)}
top_long_frames = []
for feature in top_features:
    j = feature_position[feature]
    part = sample_metadata[base_columns + ["pred_prob"]].copy()
    part["feature"] = feature
    part["feature_value"] = sample_values[:, j]
    part["shap_value"] = sample_shap[:, j]
    top_long_frames.append(part)
top_feature_values = pd.concat(top_long_frames, ignore_index=True)
top_feature_values.to_parquet(PROCESSED_ROOT / "OOF_SHAP_TOP_30_FEATURE_VALUES_V1.parquet", index=False)

display(source_importance)
display(feature_importance.head(40)[[
    "rank_oof_shap", "feature", "source_group", "feature_family",
    "mean_abs_oof_shap_incident_balanced", "oof_shap_fraction",
    "fold_rank_mean", "fold_rank_sd", "top_20_fold_count", "rank_gain",
]])

## 5. Integrate 05G and 05H robustness results

The historical-prevalence prior is constant within a test year, so its joint calibration intercept and slope are not identifiable. Those fields are set to missing for publication output. A rolling-origin aggregate combines the 2021–2024 forward predictions while retaining the fact that each year used a different historical training window.

In [ ]:
proxy_metrics = pd.read_csv(PROXY_METRICS_PATH)
temporal_metrics = pd.read_csv(TEMPORAL_METRICS_PATH)
temporal_bootstrap = pd.read_csv(TEMPORAL_BOOTSTRAP_PATH)
temporal_predictions = pd.read_parquet(TEMPORAL_PREDICTIONS_PATH)

temporal_metrics_clean = temporal_metrics.copy()
temporal_metrics_clean["calibration_estimable"] = temporal_metrics_clean["model_id"] != "HISTORICAL_PREVALENCE_PRIOR"
prior_mask = ~temporal_metrics_clean["calibration_estimable"]
for c in ["calibration_joint_intercept", "calibration_slope"]:
    temporal_metrics_clean.loc[prior_mask, c] = np.nan
temporal_metrics_clean.loc[prior_mask, "joint_converged"] = False
temporal_metrics_clean.to_csv(OUTPUT_ROOT / "TEMPORAL_METRICS_PUBLICATION_CLEAN_V1.csv", index=False)

def basic_summary(frame):
    y_true = frame["hold_label"].to_numpy(int)
    pred = clip_prob(frame["pred_prob"].to_numpy(float))
    inc_rows = []
    for _, g in frame.groupby("_irwin_norm", sort=False):
        yi = g["hold_label"].to_numpy(int)
        pi = clip_prob(g["pred_prob"].to_numpy(float))
        mixed = np.unique(yi).size == 2
        inc_rows.append({
            "brier": float(np.mean((pi - yi) ** 2)),
            "log_loss": float(log_loss(yi, pi, labels=[0, 1])),
            "mixed": mixed,
            "pr_auc": float(average_precision_score(yi, pi)) if mixed else np.nan,
            "roc_auc": float(roc_auc_score(yi, pi)) if mixed else np.nan,
        })
    inc = pd.DataFrame(inc_rows)
    mixed = inc["mixed"]
    return {
        "units": len(frame),
        "incidents": frame["_irwin_norm"].nunique(),
        "held_rate": float(np.mean(y_true)),
        "mean_pred": float(np.mean(pred)),
        "pooled_pr_auc": float(average_precision_score(y_true, pred)),
        "pooled_roc_auc": float(roc_auc_score(y_true, pred)),
        "pooled_brier": float(brier_score_loss(y_true, pred)),
        "pooled_log_loss": float(log_loss(y_true, pred, labels=[0, 1])),
        "incident_macro_brier": float(inc["brier"].mean()),
        "incident_macro_log_loss": float(inc["log_loss"].mean()),
        "mixed_incident_macro_pr_auc": float(inc.loc[mixed, "pr_auc"].mean()),
        "mixed_incident_macro_roc_auc": float(inc.loc[mixed, "roc_auc"].mean()),
    }

rolling_columns = {
    "FORWARD_FIXED_05C": "pred_forward_fixed_05c",
    "HISTORICAL_PREVALENCE_PRIOR": "pred_historical_prevalence_prior",
    "MIXED_YEAR_PRIMARY_OOF_REFERENCE": "pred_mixed_year_oof_reference",
}
rolling_rows = []
for model_id, pred_col in rolling_columns.items():
    evaluated = temporal_predictions[["unit_uid", "_irwin_norm", "Year", "hold_label"]].copy()
    evaluated["pred_prob"] = temporal_predictions[pred_col].to_numpy(float)
    rolling_rows.append({"model_id": model_id, **basic_summary(evaluated)})
rolling_metrics = pd.DataFrame(rolling_rows)
rolling_metrics.to_csv(OUTPUT_ROOT / "ROLLING_ORIGIN_AGGREGATE_METRICS_V1.csv", index=False)

display(rolling_metrics)
display(proxy_metrics)

## 6. Publication-oriented interpretation figures

Figures are exported as both PNG and vector PDF. Long feature names are retained so each plotted quantity is traceable to the frozen feature manifest. The SHAP summary uses percentile color within each feature because predictors have different physical units.

In [ ]:
GROUP_COLORS = {
    "fire_approach_physical": "#c44e52",
    "observation_process": "#8172b3",
    "weather": "#4c72b0",
    "terrain": "#55a868",
    "fuels_canopy": "#dd8452",
}

def save_figure(fig, stem):
    png = FIGURE_ROOT / f"{stem}.png"
    pdf = FIGURE_ROOT / f"{stem}.pdf"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    return png, pdf

# Figure 1: source-group importance.
g = source_importance.sort_values("oof_shap_fraction")
fig, ax = plt.subplots(figsize=(8.5, 4.8), constrained_layout=True)
ax.barh(g["source_group"], g["oof_shap_fraction"], color=[GROUP_COLORS[x] for x in g["source_group"]])
ax.set_xlabel("Fraction of incident-balanced mean |OOF SHAP|")
ax.set_ylabel("Predictor source")
ax.set_title("Held-out predictive importance by source")
ax.grid(axis="x", alpha=0.25)
save_figure(fig, "FIG_05I_01_SOURCE_GROUP_OOF_SHAP_V1")
plt.show()

# Figure 2: top individual features.
top_bar = feature_importance.head(TOP_FEATURES_BAR).sort_values("mean_abs_oof_shap_incident_balanced")
fig, ax = plt.subplots(figsize=(11, 10), constrained_layout=True)
ax.barh(top_bar["feature"], top_bar["mean_abs_oof_shap_incident_balanced"], color=[GROUP_COLORS[x] for x in top_bar["source_group"]])
ax.set_xlabel("Incident-balanced mean |OOF SHAP| (log-odds)")
ax.set_title("Top held-out predictive features")
ax.grid(axis="x", alpha=0.25)
save_figure(fig, "FIG_05I_02_TOP_FEATURE_OOF_SHAP_V1")
plt.show()

# Figure 3: SHAP distribution and within-feature value percentile.
summary_features = feature_importance.head(TOP_FEATURES_SUMMARY)["feature"].tolist()
fig, ax = plt.subplots(figsize=(11, 8.5), constrained_layout=True)
plot_rng = np.random.default_rng(SHAP_SEED + 99)
for y_pos, feature in enumerate(reversed(summary_features)):
    j = feature_position[feature]
    valid = np.flatnonzero(np.isfinite(sample_values[:, j]))
    if len(valid) > 2500:
        valid = plot_rng.choice(valid, size=2500, replace=False)
    values_here = pd.Series(sample_values[valid, j]).rank(pct=True).to_numpy(float)
    jitter = plot_rng.normal(0.0, 0.09, size=len(valid))
    ax.scatter(sample_shap[valid, j], y_pos + jitter, c=values_here, cmap="coolwarm", norm=Normalize(0, 1), s=8, alpha=0.45, linewidths=0)
ax.axvline(0.0, color="black", linewidth=0.8)
ax.set_yticks(range(len(summary_features)))
ax.set_yticklabels(list(reversed(summary_features)))
ax.set_xlabel("Held-out SHAP contribution (log-odds)")
ax.set_title("Held-out SHAP distributions for leading predictors")
sm = plt.cm.ScalarMappable(cmap="coolwarm", norm=Normalize(0, 1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.01)
cbar.set_label("Within-feature value percentile")
ax.grid(axis="x", alpha=0.2)
save_figure(fig, "FIG_05I_03_OOF_SHAP_SUMMARY_V1")
plt.show()

# Figure 4: fold-rank stability.
stability_features = feature_importance.head(20)["feature"].tolist()
rank_matrix = feature_importance.set_index("feature").loc[stability_features, [f"rank_fold_{i}" for i in range(OUTER_FOLDS)]].to_numpy(float)
fig, ax = plt.subplots(figsize=(8.5, 9.5), constrained_layout=True)
im = ax.imshow(rank_matrix, aspect="auto", cmap="viridis_r", vmin=1, vmax=min(150, np.nanmax(rank_matrix)))
ax.set_xticks(range(OUTER_FOLDS), [f"Fold {i}" for i in range(OUTER_FOLDS)])
ax.set_yticks(range(len(stability_features)), stability_features)
ax.set_title("Outer-fold rank stability of leading features")
cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label("Within-fold importance rank (lower is stronger)")
save_figure(fig, "FIG_05I_04_FEATURE_RANK_STABILITY_V1")
plt.show()

# Figure 5: 05G and 05H robustness synthesis.
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
proxy_order = ["PRIMARY_FULL", "PHYSICAL_SENSITIVITY", "PROCESS_ONLY"]
proxy_labels = ["Primary full", "Physical only", "Process only"]
p = proxy_metrics.set_index("experiment_id").loc[proxy_order]
axes[0, 0].bar(proxy_labels, p["roc_auc"], color=["#4c72b0", "#55a868", "#8172b3"])
axes[0, 0].set_ylabel("Pooled ROC-AUC")
axes[0, 0].set_title("Observation-process sensitivity")
axes[0, 0].tick_params(axis="x", rotation=15)
axes[0, 1].bar(proxy_labels, p["incident_macro_brier"], color=["#4c72b0", "#55a868", "#8172b3"])
axes[0, 1].set_ylabel("Incident-macro Brier")
axes[0, 1].set_title("Probability loss (lower is better)")
axes[0, 1].tick_params(axis="x", rotation=15)
temporal_labels = {"FORWARD_FIXED_05C": "Forward fixed 05C", "HISTORICAL_PREVALENCE_PRIOR": "Historical prior", "MIXED_YEAR_PRIMARY_OOF_REFERENCE": "Mixed-year OOF reference"}
temporal_colors = {"FORWARD_FIXED_05C": "#c44e52", "HISTORICAL_PREVALENCE_PRIOR": "#8c8c8c", "MIXED_YEAR_PRIMARY_OOF_REFERENCE": "#4c72b0"}
for model_id in temporal_labels:
    q = temporal_metrics_clean[temporal_metrics_clean["model_id"] == model_id].sort_values("test_year")
    axes[1, 0].plot(q["test_year"], q["mixed_incident_macro_roc_auc"], marker="o", linewidth=2, label=temporal_labels[model_id], color=temporal_colors[model_id])
    axes[1, 1].plot(q["test_year"], q["incident_macro_brier"], marker="o", linewidth=2, label=temporal_labels[model_id], color=temporal_colors[model_id])
axes[1, 0].set_title("Forward-year mixed-incident discrimination")
axes[1, 0].set_ylabel("Mixed-incident macro ROC-AUC")
axes[1, 1].set_title("Forward-year incident-balanced probability loss")
axes[1, 1].set_ylabel("Incident-macro Brier")
for ax in axes[1, :]:
    ax.set_xlabel("Test year")
    ax.set_xticks([2021, 2022, 2023, 2024])
    ax.grid(alpha=0.25)
axes[1, 0].legend(frameon=False, fontsize=9)
fig.suptitle("Frozen-model robustness synthesis", fontsize=15)
save_figure(fig, "FIG_05I_05_ROBUSTNESS_SYNTHESIS_V1")
plt.show()

# Figure 6: full-data gain versus held-out SHAP.
fig, ax = plt.subplots(figsize=(8, 6.5), constrained_layout=True)
for group, q in feature_importance.groupby("source_group"):
    ax.scatter(q["importance_gain_fraction"], q["oof_shap_fraction"], s=18, alpha=0.6, label=group, color=GROUP_COLORS[group])
ax.set_xscale("symlog", linthresh=1e-7)
ax.set_yscale("symlog", linthresh=1e-7)
ax.set_xlabel("Full-data gain-importance fraction")
ax.set_ylabel("Incident-balanced OOF SHAP fraction")
ax.set_title("Full-data gain versus held-out SHAP importance")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.2)
save_figure(fig, "FIG_05I_06_GAIN_VS_OOF_SHAP_V1")
plt.show()

## 7. Integrity checks and report

In [ ]:
expected_source_counts = {"fuels_canopy": 547, "weather": 148, "fire_approach_physical": 60, "terrain": 37, "observation_process": 11}
observed_source_counts = feature_metadata["source_group"].value_counts().to_dict()
figure_pngs = sorted(FIGURE_ROOT.glob("FIG_05I_*.png"))
figure_pdfs = sorted(FIGURE_ROOT.glob("FIG_05I_*.pdf"))
checks = pd.DataFrame([
    {"check": "exact_803_frozen_features", "pass": len(features) == 803},
    {"check": "exact_saved_primary_parameter_contract", "pass": {k: v for k, v in saved_primary_params.items() if k != "random_state"} == FIXED_PARAMS},
    {"check": "subsample_freq_remains_absent", "pass": "subsample_freq" not in FIXED_PARAMS},
    {"check": "exact_five_outer_refits", "pass": len(runtime) == OUTER_FOLDS},
    {"check": "all_shap_units_outer_fold_held_out", "pass": bool((sample_metadata["outer_fold"].isin(range(OUTER_FOLDS))).all())},
    {"check": "all_381_incidents_represented_in_shap_sample", "pass": sample_metadata["_irwin_norm"].nunique() == EXPECTED_INCIDENTS},
    {"check": "shap_sample_unique_units", "pass": sample_metadata["unit_uid"].is_unique},
    {"check": "maximum_50_shap_units_per_incident", "pass": bool((sample_metadata.groupby("_irwin_norm").size() <= SHAP_MAX_UNITS_PER_INCIDENT).all())},
    {"check": "regenerated_oof_matches_saved_05c", "pass": oof_reproduction_max_abs_error <= OOF_REPRODUCTION_ATOL},
    {"check": "tree_shap_additivity_passes", "pass": max_shap_additivity_error <= SHAP_ADDITIVITY_ATOL},
    {"check": "all_shap_values_finite", "pass": bool(np.isfinite(sample_shap).all())},
    {"check": "all_features_have_exactly_one_source_group", "pass": observed_source_counts == expected_source_counts},
    {"check": "all_11_observation_process_features_separate", "pass": observed_source_counts.get("observation_process") == 11},
    {"check": "feature_importance_exact_803_rows", "pass": len(feature_importance) == 803},
    {"check": "temporal_prior_calibration_marked_not_estimable", "pass": bool((~temporal_metrics_clean.loc[prior_mask, "calibration_estimable"]).all() and temporal_metrics_clean.loc[prior_mask, "calibration_slope"].isna().all())},
    {"check": "exact_six_png_and_six_pdf_figures", "pass": len(figure_pngs) == 6 and len(figure_pdfs) == 6},
    {"check": "no_hyperparameter_tuning", "pass": True},
    {"check": "no_feature_selection_or_primary_model_change", "pass": True},
    {"check": "no_calibration_or_threshold_selection", "pass": True},
    {"check": "interpretation_explicitly_noncausal", "pass": True},
])
PASS = bool(checks["pass"].all())
checks.to_csv(OUTPUT_ROOT / "FINAL_FROZEN_MODEL_INTERPRETATION_CHECKS_V1.csv", index=False)
display(checks)

def md_table(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

report_lines = [
    "# 05I Frozen-Model Interpretation and Publication Figures",
    "",
    f"Build: `{NOTEBOOK_BUILD_ID}`",
    "",
    f"Generated UTC: {datetime.now(timezone.utc).isoformat()}",
    "",
    "## Scientific status",
    "",
    "Interpretation of the frozen 05C primary model. No model selection, tuning, calibration fitting, or threshold selection occurred.",
    "",
    "All SHAP results are predictive associations from outer-fold-held-out units, aggregated equally across wildfire incidents. They are not causal effects.",
    "",
    "## OOF reproduction and SHAP audit",
    "",
    f"Saved-05C OOF maximum absolute probability difference: `{oof_reproduction_max_abs_error:.12g}`",
    "",
    f"TreeSHAP maximum additivity probability error: `{max_shap_additivity_error:.12g}`",
    "",
    f"Incident-balanced SHAP sample: `{len(sample_metadata):,}` units from `{sample_metadata['_irwin_norm'].nunique():,}` incidents.",
    "",
    "## Source-group importance",
    "",
    md_table(source_importance),
    "",
    "## Feature-family importance",
    "",
    md_table(family_importance),
    "",
    "## Top 40 held-out SHAP features",
    "",
    md_table(feature_importance.head(40)),
    "",
    "## 05G observation-process sensitivity",
    "",
    md_table(proxy_metrics),
    "",
    "## Rolling-origin aggregate metrics",
    "",
    md_table(rolling_metrics),
    "",
    "## Forward temporal metrics",
    "",
    md_table(temporal_metrics_clean),
    "",
    "## Final checks",
    "",
    md_table(checks),
    "",
    "## Decision",
    "",
    "`PASS / FROZEN MODEL INTERPRETATION COMPLETE; READY FOR MANUSCRIPT ASSEMBLY`" if PASS else "`REVIEW REQUIRED`",
]
REPORT_PATH = OUTPUT_ROOT / "FROZEN_MODEL_INTERPRETATION_REPORT_V1.md"
REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")

print()
print("05I FROZEN-MODEL INTERPRETATION:", "PASS" if PASS else "REVIEW REQUIRED")
print("Saved:", REPORT_PATH)
print("DONE:", NOTEBOOK_BUILD_ID)

## 8. Output inventory

In [ ]:
print("=" * 100)
print("05I OUTPUTS")
print("=" * 100)
for root in [OUTPUT_ROOT, PROCESSED_ROOT]:
    print()
    print(root)
    for p in sorted(root.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(root)!s:<85} {p.stat().st_size/(1024**2):>10.3f} MB")